In [ ]:
<h4 style="color:white;text-align:center; background-color:#4CAF50; padding:5px; border-radius:5px;">
Purpose of Content Based Recommendation
</h4>

In [15]:
from IPython.display import display, HTML

html_table = """
<div style="font-family:Arial; width:600px; margin:auto;">

<h3 style="
    text-align:center;
    background-color:#4CAF50;
    color:white;
    padding:8px;
    border-radius:6px;
">
Content-Based Recommendation
</h3>
<h4>Input = User → Output = Movies → Purpose = Based on user preferences
<table style="
    width:100%;
    border-collapse: collapse;
    margin-top:10px;
">

    <tr style="background-color:#E8F5E9;">
        <th style="padding:10px; border:1px solid #ccc;">Section</th>
        <th style="padding:10px; border:1px solid #ccc;">Details</th>
    </tr>

    <tr>
        <td style="padding:10px; border:1px solid #ccc;"><b>Input</b></td>
        <td style="padding:10px; border:1px solid #ccc;">User ID = 54</td>
    </tr>

    <tr style="background-color:#f9f9f9;">
        <td style="padding:10px; border:1px solid #ccc;"><b>Output</b></td>
        <td style="padding:10px; border:1px solid #ccc;">Iron Man, Thor, Die Hard</td>
    </tr>

    <tr>
        <td style="padding:10px; border:1px solid #ccc;"><b>Purpose</b></td>
        <td style="padding:10px; border:1px solid #ccc;">
        Recommend movies based on user's preferred genres
        </td>
    </tr>
    
</table>

</div>

"""

display(HTML(html_table))

Section,Details
Input,User ID = 54
Output,"Iron Man, Thor, Die Hard"
Purpose,Recommend movies based on user's preferred genres


#### Load Dataset

In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load movie dataset
i_cols = [
    'movie id', 'movie title', 'release date', 'video release date', 'IMDb URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
    'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

items = pd.read_csv('ml-100k/u.item',sep='|',names=i_cols,encoding='latin-1')

# Select only required columns (features)
movie_content = items[['movie id', 'movie title',
    'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
    'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]]

# Load ratings dataset
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']

ratings = pd.read_csv('ml-100k/u.data',sep='\t',names=r_cols, encoding='latin-1'
)

In [16]:
def content_based_recommendation(ratings, movie_content, user_id, top_n=10):

    #  Step 1: Get movies watched by user
    watched_movies = ratings[ratings['user_id'] == user_id]['movie_id'].values

    
    #  Step 2: Get feature vectors of watched movies
    user_profile = movie_content[
        movie_content['movie id'].isin(watched_movies)
    ].drop(columns=['movie id', 'movie title'])


    # Step 3: Create user preference vector (average of watched movies)
    user_profile_vector = user_profile.mean().values.reshape(1, -1)


    # Step 4: Get all movie feature vectors
    all_movie_vectors = movie_content.drop(columns=['movie id', 'movie title'])


    # Step 5: Compute similarity between user profile and all movies
    similarity_scores = cosine_similarity(user_profile_vector, all_movie_vectors)[0]


    # Step 6: Add similarity scores to dataframe
    movie_content.loc[:, 'similarity'] = similarity_scores


    # Step 7: Remove already watched movies
    recommendations = movie_content[
        ~movie_content['movie id'].isin(watched_movies)
    ]


    # Step 8: Sort by similarity and select top movies
    recommendations = recommendations.sort_values(
        by='similarity', ascending=False
    ).head(top_n)


    # Step 9: Return movie IDs or titles
    return recommendations[['movie id', 'movie title', 'similarity']]

In [17]:
result = content_based_recommendation(ratings=ratings,movie_content=movie_content,
                                    user_id=54,top_n=10)

print(result)


      movie id                    movie title  similarity
916        917          Mercury Rising (1998)    0.880245
1558      1559      Hostile Intentions (1994)    0.880245
1555      1556           Condition Red (1995)    0.880245
53          54                Outbreak (1995)    0.880245
27          28               Apollo 13 (1995)    0.880245
1490      1491        Tough and Deadly (1995)    0.880245
1024      1025         Fire Down Below (1997)    0.880245
243        244  Smilla's Sense of Snow (1997)    0.880245
1            2               GoldenEye (1995)    0.825873
1104      1105               Firestorm (1998)    0.825873
